# Customer Churn Prediction — Exploratory Data Analysis
**Author:** Aleena Anam | [GitHub](https://github.com/anam-aleena)

This notebook covers:
- Dataset overview and descriptive statistics
- Churn distribution analysis
- Feature correlation with churn
- Visual insights driving feature engineering decisions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')
from pipeline import generate_churn_data

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

df = generate_churn_data(5000)
print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Descriptive Statistics ---')
df.describe().round(2)

## 2. Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df['Churn'].value_counts()
axes[0].bar(['Retained', 'Churned'], churn_counts.values, color=['#2196F3','#F44336'], edgecolor='white')
axes[0].set_title('Churn Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie
axes[1].pie(churn_counts.values, labels=['Retained','Churned'],
            autopct='%1.1f%%', colors=['#2196F3','#F44336'], startangle=90)
axes[1].set_title('Churn Rate', fontsize=13, fontweight='bold')

plt.suptitle('Overall Churn Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Churn Rate: {df["Churn"].mean():.1%}')

## 3. Numeric Features vs Churn

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'NumServices']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df[df['Churn']==0][col].hist(ax=axes[i], alpha=0.6, bins=30, color='#2196F3', label='Retained')
    df[df['Churn']==1][col].hist(ax=axes[i], alpha=0.6, bins=30, color='#F44336', label='Churned')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend()

plt.suptitle('Numeric Features by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/numeric_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Categorical Features vs Churn

In [ ]:
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    bars = axes[i].barh(churn_rate.index, churn_rate.values * 100,
                        color=['#F44336' if v > 0.27 else '#2196F3' for v in churn_rate.values])
    axes[i].set_title(f'Churn Rate by {col}', fontweight='bold')
    axes[i].set_xlabel('Churn Rate (%)')
    for bar, val in zip(bars, churn_rate.values):
        axes[i].text(val*100 + 0.3, bar.get_y() + bar.get_height()/2,
                     f'{val:.1%}', va='center', fontsize=10)

plt.suptitle('Churn Rate by Categorical Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/categorical_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
from sklearn.preprocessing import LabelEncoder
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col])

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_enc.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Insights
- **Month-to-month contracts** have the highest churn rate — strongest single predictor
- **Short tenure (< 12 months)** customers churn at significantly higher rates
- **Fiber optic internet** users churn more than DSL users
- **Electronic check** payment method correlates with higher churn
- **TechSupport = No** customers are more likely to churn
- These insights directly informed our feature selection and engineering strategy